# Distilled model vs lexicon rules, on studies neither has seen

**The question.** The atlas grows. You label a batch of contrasts once, then want new
studies categorised without supervision. Is the distilled model good enough to trust
unsupervised, and is it better than the deterministic rules?

**Why grouped by study.** Contrasts within a GEO series share nearly all their text. A random
row split puts the same series on both sides, so the model recognises the series rather than
the biology. Every fold here holds out whole studies — exactly the deployment situation.

**What this measures.** Gold labels are the training target, so this is the model's *ceiling*:
in production it learns from LLM teacher labels that carry their own error.

This notebook calls the same functions as `compare_classifiers.py`, so the logic cannot drift.


In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import contrast_features as cfx
from compare_classifiers import (load_gold, run_rules, grouped_model_predictions,
                                 selective_curve, mcnemar_exact)

COMPARISONS = 'comparison_registry.csv'   # your registry
GOLD        = 'gold_corrected.csv'        # your reviewed labels
CACHE       = 'geo_cache'
SEED, FOLDS = 0, 5

## 1. Assemble: features, gold labels, and one rules pass

The rules need no training, so they run once over everything and are scored per fold. That is not leakage — nothing about the held-out studies influenced them.

In [ ]:
reg = pd.read_csv(COMPARISONS)
if 'technology' not in reg.columns:
    reg['technology'] = 'unknown'

gold  = load_gold(GOLD)
rules = run_rules(COMPARISONS, CACHE, '.')
X     = cfx.build_features(reg, CACHE)

df = X.merge(gold, on=['study_id', 'comparison_id'], how='inner') \
      .merge(rules, on=['study_id', 'comparison_id'], how='left')
df = df[df.rules_category.notna()].reset_index(drop=True)
print('%d contrasts across %d studies' % (len(df), df.study_id.nunique()))
df.perturbation_category.value_counts()

## 2. Out-of-fold predictions, whole studies held out

In [ ]:
truth = df.perturbation_category.to_numpy()
rpred = df.rules_category.to_numpy()
pred, prob, folds = grouped_model_predictions(df[X.columns], truth,
                                              df.study_id.to_numpy(), SEED, FOLDS)

print('distilled model (grouped CV) : %.4f' % (pred  == truth).mean())
print('lexicon rules                : %.4f' % (rpred == truth).mean())

## 3. Is the difference real?

Exact McNemar on the discordant pairs — the paired test for two classifiers scored on the same contrasts. Concordant pairs carry no information about which is better and are excluded by construction.

In [ ]:
mc = mcnemar_exact(truth, pred, rpred)
print(mc)

## 4. Selective prediction — the operating question

"Unsupervised" need not mean "accept everything". The model emits a probability, so the real
question is what fraction can be auto-accepted, at what accuracy, and how many are left to
review.

If accuracy on the accepted set *rises* as the threshold rises, the model is calibrated and a
threshold buys you something. If it does not, confidence is uninformative and there is no safe
operating point.

In [ ]:
curve = selective_curve(truth, pred, prob, np.arange(0.0, 0.96, 0.05))
curve[curve.auto_accepted > 0]

In [ ]:
d = curve[curve.auto_accepted > 0]
rules_acc = float((rpred == truth).mean())

fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.axhline(rules_acc, color='#1b7837', lw=1.6)
ax.text(0.02, rules_acc + 0.025, 'lexicon rules, all contrasts (%.2f)' % rules_acc,
        color='#1b7837', fontsize=7)
sc = ax.scatter(d.coverage, d.accuracy_on_accepted, s=26 + 130 * d.coverage,
                c=d.threshold, cmap='viridis', edgecolor='white', lw=0.6, zorder=4)
ax.plot(d.coverage, d.accuracy_on_accepted, color='#4a4a4a', lw=1.0, alpha=0.55)
ax.set_xlabel('Coverage — fraction of contrasts auto-accepted')
ax.set_ylabel('Accuracy among auto-accepted')
fig.colorbar(sc, ax=ax, pad=0.02).set_label('probability threshold', fontsize=6)
fig.tight_layout()

## 5. Where the model loses

Contrasts the rules got right and the model did not. If these concentrate in one category or
one kind of contrast name, that is a lexicon the model lacks rather than a general weakness.

In [ ]:
lost = df.assign(truth=truth, model=pred, rules=rpred, p=prob.round(3))
lost = lost[(lost.rules == lost.truth) & (lost.model != lost.truth)]
print('%d contrasts: rules right, model wrong' % len(lost))
lost.groupby(['truth', 'model']).size().sort_values(ascending=False).head(10)

## 6. Reading the result

- If the rules win decisively and the coverage/accuracy curve is flat or falling, deploy the
  rules for unsupervised categorisation and keep the model out of the loop. The lexicon CSVs
  are the tuning surface; there is no accuracy cliff on unseen studies because nothing was
  learned from study vocabulary.
- If the model wins, or the curve rises steeply, pick the threshold where accuracy on the
  accepted set clears your tolerance and route the remainder to review.

Either way, re-run this whenever the atlas grows — the answer depends on how much labelled
data exists, and it can change.